# BaseEvaluator — code walkthrough

Step through each method with real data.

In [1]:
import json
from pathlib import Path

path = Path(r"D:\J\Desktop\language_technology\course\projects_AI\mt_oil_no\outputs\exp1_data_scaling\size_13935\seed_42\test_predictions.json")

with open(path, encoding="utf-8") as f:
    data = json.load(f)

sources     = [d["source"]    for d in data]
references  = [d["reference"] for d in data]
predictions = [d["prediction"] for d in data]

print(len(data), "samples")
print()
print("source    :", sources[0])
print("reference :", references[0])
print("prediction:", predictions[0])

1742 samples

source    : Pressure gradients were not established and the forecasted Permian-Triassic conglomerates were not encountered.
reference : Trykkgradienter ble ikke etablert og de prognoserte konglomeratene av perm-trias alder ble ikke påtruffet.
prediction: Trykkgradienter ble ikke etablert og de prognoserte perm-trias konglomeratene ble ikke påtruffet.


---
## `__init__` — what gets loaded at startup

In [2]:
import evaluate

# these two lines load the metric objects from HuggingFace
bleu = evaluate.load("bleu")
chrf = evaluate.load("chrf")

# comet is NOT loaded yet — only loaded if use_comet=True
# because it takes ~30 seconds and downloads a neural model
print(type(bleu))
print(type(chrf))

d:\envs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<class 'evaluate_modules.metrics.evaluate-metric--bleu.9e0985c1200e367cce45605ce0ecb5ede079894e0f24f54613fca08eeb8aff76.bleu.Bleu'>
<class 'evaluate_modules.metrics.evaluate-metric--chrf.d244bab9383988714085a8dacc4871986d9f025398581c33d6b2ee22836b4069.chrf.ChrF'>


---
## `compute_bleu()` — line by line

In [3]:
# this is what the function calls internally
result = bleu.compute(
    predictions=predictions,
    references=[[ref] for ref in references],  # wrapped in list — BLEU supports multiple refs per sentence
)

# see everything that comes back
result

{'bleu': 0.618657368177976,
 'precisions': [0.7983146798542425,
  0.6567910602193557,
  0.5650706292423409,
  0.4944207352883599],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0066155760791249,
 'translation_length': 30736,
 'reference_length': 30534}

In [4]:
# the function then picks out what it needs
precisions = result["precisions"]
print("precisions:", precisions)
# precisions[0] = 1-gram, [1] = 2-gram, [2] = 3-gram, [3] = 4-gram

precisions: [0.7983146798542425, 0.6567910602193557, 0.5650706292423409, 0.4944207352883599]


In [5]:
# final return value of compute_bleu()
{
    "bleu":   result["bleu"],
    "bleu_1": precisions[0] if precisions else 0.0,
    "bleu_2": precisions[1] if len(precisions) > 1 else 0.0,
    "bleu_3": precisions[2] if len(precisions) > 2 else 0.0,
    "bleu_4": precisions[3] if len(precisions) > 3 else 0.0,
}

{'bleu': 0.618657368177976,
 'bleu_1': 0.7983146798542425,
 'bleu_2': 0.6567910602193557,
 'bleu_3': 0.5650706292423409,
 'bleu_4': 0.4944207352883599}

In [6]:
# why [[ref] for ref in references] and not just references?
print("wrong:", references[:2])
print("right:", [[ref] for ref in references[:2]])

# BLEU allows multiple valid references per sentence
# e.g. [["ref A", "ref B"], ["ref C", "ref D"]]
# we only have one so we wrap each in a list

wrong: ['Trykkgradienter ble ikke etablert og de prognoserte konglomeratene av perm-trias alder ble ikke påtruffet.', 'Utvinnbare ressurser er beregnet til 2-4 millioner Sm3 o.e.']
right: [['Trykkgradienter ble ikke etablert og de prognoserte konglomeratene av perm-trias alder ble ikke påtruffet.'], ['Utvinnbare ressurser er beregnet til 2-4 millioner Sm3 o.e.']]


---
## `compute_chrf()` — line by line

In [7]:
# chrf is simpler — references are flat strings, not wrapped
result = chrf.compute(
    predictions=predictions,
    references=references,
)

# see everything that comes back
result

{'score': 79.19612374541678, 'char_order': 6, 'word_order': 0, 'beta': 2}

In [8]:
# the function only keeps the score
{"chrf": result["score"]}

{'chrf': 79.19612374541678}

In [9]:
# key difference from BLEU: chrF works at character level
# so partial word matches still count
# example: prediction has 'konglomeratene' but reference has 'konglomeratene av perm-trias alder'

p = [predictions[0]]
r = [references[0]]

b = bleu.compute(predictions=p, references=[r])
c = chrf.compute(predictions=p, references=r)

print("pred:", p[0])
print("ref :", r[0])
print()
print("BLEU:", round(b["bleu"], 4))   # word-level mismatch penalised heavily
print("chrF:", round(c["score"], 4))  # character overlap still gives partial credit

pred: Trykkgradienter ble ikke etablert og de prognoserte perm-trias konglomeratene ble ikke påtruffet.
ref : Trykkgradienter ble ikke etablert og de prognoserte konglomeratene av perm-trias alder ble ikke påtruffet.

BLEU: 0.5993
chrF: 85.1818


---
## `compute_comet()` — line by line

In [10]:
from comet import download_model, load_from_checkpoint

comet_model = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da"))
print("loaded")

ModuleNotFoundError: No module named 'comet'

In [ ]:
# the function first builds a list of dicts
# COMET is the only metric that uses the source sentence
comet_data = [
    {"src": src, "mt": pred, "ref": ref}
    for src, pred, ref in zip(sources, predictions, references)
]

print("example input:")
print(json.dumps(comet_data[0], indent=2, ensure_ascii=False))

In [ ]:
# then runs prediction
result = comet_model.predict(
    comet_data,
    batch_size=4,
    accelerator="auto",   # auto-detects GPU or CPU
)

# see what comes back
print("system_score:", result["system_score"])   # mean across all sentences
print("scores[:5]  :", result["scores"][:5])     # per-sentence scores

In [ ]:
import numpy as np

# final return value of compute_comet()
{
    "comet":     float(result["system_score"]),        # overall score
    "comet_std": float(np.std(result["scores"])),      # how much variance across sentences
}

In [ ]:
# the function also has a guard at the top
# if comet wasn't loaded (use_comet=False or loading failed)
# it returns None immediately without crashing

use_comet = False
comet_model_check = None

if not use_comet or comet_model_check is None:
    print("returning early:", {"comet": None, "comet_std": None})

---
## `evaluate_all()` — puts it all together

In [ ]:
# step 1: length checks
print(len(predictions) == len(references))   # must match
print(len(sources) == len(predictions))      # must match if using comet

In [ ]:
# step 2: strip whitespace from all strings
sources_clean     = [s.strip() for s in sources]
predictions_clean = [p.strip() for p in predictions]
references_clean  = [r.strip() for r in references]

# check first item before and after
print(repr(predictions[0][:30]))
print(repr(predictions_clean[0][:30]))

In [ ]:
# step 3: collect metrics one by one into a single dict
metrics = {}

bleu_scores = {
    "bleu":   result["bleu"] if "bleu" in result else bleu.compute(predictions=predictions_clean, references=[[r] for r in references_clean])["bleu"],
}

# actually just run compute_bleu
r_bleu = bleu.compute(predictions=predictions_clean, references=[[r] for r in references_clean])
metrics.update({
    "bleu":   r_bleu["bleu"],
    "bleu_1": r_bleu["precisions"][0],
    "bleu_2": r_bleu["precisions"][1],
    "bleu_3": r_bleu["precisions"][2],
    "bleu_4": r_bleu["precisions"][3],
})

r_chrf = chrf.compute(predictions=predictions_clean, references=references_clean)
metrics.update({"chrf": r_chrf["score"]})

metrics.update({
    "comet":     float(result["system_score"]),
    "comet_std": float(np.std(result["scores"])),
})

# final output
metrics